# MACE MLIP fine-tuning script walkthrough

This notebook explains `../examples/mace_mlip_training.sh`, a scheduler-neutral version of the MACE MLIP fine-tuning workflow. The script is designed to be run after you activate an environment that contains `mace-torch`, PyTorch, ASE, and any other project dependencies.

## Documentation referenced

MACE evolves quickly, especially around foundation-model fine-tuning. The notes below were written against the current MACE documentation checked on 2026-05-11:

- Training guide: https://mace-docs.readthedocs.io/en/latest/guide/training.html
- Fine-tuning foundation models: https://mace-docs.readthedocs.io/en/latest/guide/finetuning.html
- Multihead replay fine-tuning: https://mace-docs.readthedocs.io/en/latest/guide/multihead_finetuning.html
- Multi-GPU training: https://mace-docs.readthedocs.io/en/latest/guide/multigpu.html
- Foundation models: https://mace-docs.readthedocs.io/en/latest/guide/foundation_models.html
- MACE foundation-model releases and replay data: https://github.com/ACEsuit/mace-foundations/releases

In [ ]:
from pathlib import Path

script_path = Path("../examples/mace_mlip_training.sh")
print(script_path.resolve())
print(script_path.read_text())

## Workflow shape

The script has four parts:

1. Define configurable variables such as training files, foundation model, replay data, loss weights, batch size, precision, and device.
2. Check that the target training, test, and optional validation files exist.
3. Choose the launch command. A single-process run uses `python -m mace.cli.run_train`; a multi-GPU single-node run uses `torchrun` and adds `--distributed`.
4. Run MACE training and capture standard output and errors into a log file with `tee`.

This keeps the MLIP training workflow independent of a specific queueing system. If you later need Slurm, PBS, or another scheduler, wrap this script in a small scheduler submission file instead of putting site-specific details inside the workflow itself.

## Preparing an extended XYZ training file from `.traj`

The helper script `../examples/prepare_training_set.py` converts an ASE trajectory file into an extended XYZ file suitable for the MACE training scripts. It reads the structures from `.traj`, extracts stored energies and forces, then writes them as `ref_energy` and `ref_forces`.

From the repository root, convert all frames in a trajectory with:

```bash
python examples/prepare_training_set.py opt.traj training.extxyz
```

If your terminal only has `python3`, use:

```bash
python3 examples/prepare_training_set.py opt.traj training.extxyz
```

The output filename is optional. Without it, the script changes the suffix of the input file:

```bash
python examples/prepare_training_set.py opt.traj
```

This writes `opt.extxyz`.

You can select frames using ASE's normal index syntax. For example, the last frame only:

```bash
python examples/prepare_training_set.py opt.traj final_structure.extxyz --index -1
```

The first 50 frames:

```bash
python examples/prepare_training_set.py opt.traj training.extxyz --index 0:50
```

The generated file can then be passed to the minimal CPU example or the full fine-tuning script:

```bash
TRAIN_FILE=training.extxyz TEST_FILE=test.extxyz ./examples/mace_mlip_training_cpu_minimal.sh
```

If the source trajectory already stores energies or forces under project-specific names, use `--source-energy-key` and `--source-forces-key` to tell the converter where to read them from.

## Input and label variables

| Script variable | MACE flag | Meaning |
|---|---|---|
| `RUN_NAME` | `--name` | Name used by MACE for logs, checkpoints, and output files. |
| `TRAIN_FILE` | `--train_file` | Extended XYZ file containing the target fine-tuning structures. |
| `TEST_FILE` | `--test_file` | Independent test set evaluated after training. |
| `VALID_FILE` | `--valid_file` | Optional explicit validation set. If unset, the script uses `VALID_FRACTION`. |
| `VALID_FRACTION` | `--valid_fraction` | Fraction of `TRAIN_FILE` held out for validation when `VALID_FILE` is not supplied. |
| `ENERGY_KEY` | `--energy_key` | Name of the energy field in each XYZ configuration. The example uses `ref_energy`. |
| `FORCES_KEY` | `--forces_key` | Name of the forces array in each XYZ configuration. The example uses `ref_forces`. |
| `E0S` | `--E0s` | Isolated-atom reference energies keyed by atomic number. |

The MACE training guide describes `--train_file`, validation options, and `--test_file` as the standard data split controls. It also notes that energies and forces default to keys named `energy` and `forces`, so `--energy_key` and `--forces_key` are required when the XYZ file uses project-specific names such as `ref_energy` and `ref_forces`.

In this workflow, the regular XYZ metadata names `energy` and `forces` caused issues because ASE and MACE can both interpret or populate those conventional fields. To avoid clashes, the reference labels were stored under explicit metadata names, `ref_energy` and `ref_forces`, and the script points MACE to those fields with `--energy_key='ref_energy'` and `--forces_key='ref_forces'`. This keeps the DFT reference labels separate from any calculator-generated ASE values in the XYZ metadata.

For `E0S`, the current multihead fine-tuning guide recommends using appropriate reference energies for the target data rather than blindly using `average`, especially when initial errors are large. The values in the script preserve the Ti, O, and Pt workflow from the original example, but you should recalculate or replace them for any different chemistry or reference method.

## Foundation model and replay data

| Script variable | MACE flag | Meaning |
|---|---|---|
| `FOUNDATION_MODEL` | `--foundation_model` | MACE model shortcut, URL, or local `.model` path used as the starting point. |
| fixed `True` | `--multiheads_finetuning=True` | Enables multihead fine-tuning with target and replay heads. |
| `REPLAY_TRAIN_FILE` | `--pt_train_file` | Replay dataset shortcut or replay XYZ file. The original workflow used `omat`. |
| `NUM_SAMPLES_PT` | `--num_samples_pt` | Number of replay configurations sampled during direct multihead fine-tuning. |
| `FILTER_TYPE_PT` | `--filter_type_pt` | How replay structures are filtered by chemistry. `combinations` keeps matching element combinations. |
| `SUBSELECT_PT` | `--subselect_pt` | Replay subset selection method. The documentation lists `fps` and `random`. |
| `WEIGHT_PT` | `--weight_pt` | Loss weight for the pretraining or replay head. |
| `ATOMIC_NUMBERS` | `--atomic_numbers` | Atomic numbers to keep in replay selection. The example is O, Ti, Pt: `[8, 22, 78]`. |

The fine-tuning documentation describes three protocols: naive fine-tuning, multihead replay fine-tuning, and LoRA. For materials foundation models, the docs recommend multihead replay because the replay head helps reduce catastrophic forgetting while adapting the model to the target dataset.

The MACE foundation-model documentation lists MACE-OMAT-0 as a pretrained materials model trained on OMAT. The script defaults to `FOUNDATION_MODEL=medium_omat`, matching current fine-tuning examples. A local model path is also valid and is often the most reproducible choice for teaching or production work.

If your installed MACE version does not recognise `REPLAY_TRAIN_FILE=omat`, download the appropriate replay dataset from the MACE foundation-model releases and set `REPLAY_TRAIN_FILE=/path/to/replay_dataset.xyz`.

## Model size, loss, and optimiser flags

| Script variable | MACE flag | Meaning |
|---|---|---|
| `NUM_CHANNELS` | `--num_channels` | Controls model capacity. Larger values are usually more accurate but slower and more memory intensive. |
| `ENERGY_WEIGHT` | `--energy_weight` | Weight of the energy term in the loss. |
| `FORCES_WEIGHT` | `--forces_weight` | Weight of the force term in the loss. Force fitting commonly receives a larger weight for MLIPs. |
| `LR` | `--lr` | Learning rate. Multihead fine-tuning may apply its own lower default unless overridden by advanced flags. |
| `SCALING` | `--scaling` | Output scaling strategy. The original workflow uses `rms_forces_scaling`. |
| `BATCH_SIZE` | `--batch_size` | Number of training configurations per optimisation step. Reduce this if GPU memory is exhausted. |
| `VALID_BATCH_SIZE` | `--valid_batch_size` | Batch size used during validation. |
| `MAX_NUM_EPOCHS` | `--max_num_epochs` | Maximum number of passes through the training data. |
| `START_SWA` | `--start_swa` | Epoch at which stochastic weight averaging begins. |
| fixed `--swa` | `--swa` | Enables stochastic weight averaging. The MACE guide notes this usually improves energy errors late in training. |
| fixed `--ema` | `--ema` | Enables exponential moving average of weights for training stability. |
| `EMA_DECAY` | `--ema_decay` | Decay factor for EMA. |
| fixed `--amsgrad` | `--amsgrad` | Uses the AMSGrad variant of Adam. |
| `ERROR_TABLE` | `--error_table` | Format of the error table printed by MACE, here `PerAtomMAE`. |

`MAX_NUM_EPOCHS=200` preserves the longer original workflow. The multihead replay documentation notes that runs with large replay samples often converge in fewer epochs, around 10 to 30 in many cases, so use validation curves rather than treating 200 as mandatory.

`START_SWA=150` is set inside the 200-epoch window. If you shorten `MAX_NUM_EPOCHS`, update `START_SWA` as well or omit it and let MACE's defaults control SWA timing.

## Runtime flags

| Script variable | MACE flag | Meaning |
|---|---|---|
| `DEVICE` | `--device` | Runtime device. Current MACE docs list `cuda`, `cpu`, and `mps`. |
| `DEFAULT_DTYPE` | `--default_dtype` | Floating-point precision. The training guide notes that `float64` is default, while `float32` can train faster. |
| `SEED` | `--seed` | Random seed for reproducibility. |
| fixed `--restart_latest` | `--restart_latest` | Restarts from the latest checkpoint if one exists. |
| `NPROC_PER_NODE` | launcher plus `--distributed` | Set above 1 to launch a single-node multi-GPU run with `torchrun`. |
| `LOG_FILE` | shell `tee` | Captures the terminal output in a reusable log file. |

For one GPU, the script launches `python -m mace.cli.run_train`. For multiple GPUs on one node, it launches `torchrun --standalone --nnodes=1 --nproc_per_node=$NPROC_PER_NODE -m mace.cli.run_train` and adds `--distributed`, following the MACE multi-GPU documentation.

In [ ]:
example_commands = [
    "TRAIN_FILE=data/train.xyz TEST_FILE=data/test.xyz ./examples/mace_mlip_training.sh",
    "DEVICE=cpu TRAIN_FILE=data/train.xyz TEST_FILE=data/test.xyz ./examples/mace_mlip_training.sh",
    "NPROC_PER_NODE=4 TRAIN_FILE=data/train.xyz TEST_FILE=data/test.xyz ./examples/mace_mlip_training.sh",
    "FOUNDATION_MODEL=/path/to/mace-omat.model REPLAY_TRAIN_FILE=/path/to/replay.xyz TRAIN_FILE=data/train.xyz TEST_FILE=data/test.xyz ./examples/mace_mlip_training.sh",
]

for command in example_commands:
    print(command)

## Practical checks before training

- Confirm the training and test files are extended XYZ files and contain the fields named by `ENERGY_KEY` and `FORCES_KEY`.
- Confirm `E0S` matches the elements and reference method in the training data.
- Confirm `ATOMIC_NUMBERS` contains every element in the target dataset when replay selection is enabled.
- Start with a small run or fewer epochs to verify that data loading, validation, checkpointing, and logging all work.
- Inspect early validation errors. The multihead documentation warns that very high initial errors can indicate inconsistent or unsuitable E0 reference energies.
- Reduce `BATCH_SIZE`, `VALID_BATCH_SIZE`, `NUM_CHANNELS`, or `DEFAULT_DTYPE=float32` if memory is limiting. Only change precision after considering the accuracy needs of the system.

## Where outputs go

The script does not change directory or copy files to scratch. MACE writes its normal outputs relative to the directory from which you run the script, and the shell log is written to `LOG_FILE`, which defaults to `${RUN_NAME}.output.log`.

MACE also writes training checkpoints in the run directory. Because the script includes `--restart_latest`, a stopped or interrupted training job can be restarted by running the same script again from the same folder, with the same `RUN_NAME` and output/checkpoint files still present. MACE will look for the latest compatible checkpoint and continue from it.

If you want a completely fresh training run, change `RUN_NAME`, move the previous outputs elsewhere, or remove the previous checkpoint/output directory before restarting.

This is intentional: output location is a workflow choice, while cluster scratch staging is a site policy. On an HPC system, do scratch staging in the scheduler wrapper if the local file system requires it.